# Data Description
- Author: RUEI-HSIANG, CHANG（張睿庠；AI協助），B12303030
- Background: 依學生指定的臺北、高雄範圍補查G1–G6，保存搜尋、來源回應及全文人工判定，供可重現資料處理。
- date received: 20261108（本次實際取得及整理）
- Path to data file: 各原站網址見source_registry與下載manifest；本地固定路徑見input_manifest。
- Unit of observation: 每次網頁／API／PDF回應、逐位候選人搜尋記錄、候選人×變數人工觀察。
- Sample period: 2026候選人，歷史公職／學歷資料為核身輔助；收集截至2026-11-08。
- Known issues: HTTP失敗、編碼錯誤、同名錯頁、只有頁殼與下載錯正文均保留；不能將HTTP200當全文成功。AI／未核彙整頁只列線索。
- Definition for each variable:
  - candidate_key／candidate_name／county_city: 登記母表鍵、姓名與縣市；不以裸姓名跨人合併。
  - variable／value: 人工判定的G1年齡、G3分類、G5首次、G6家族值；空白維持待核。
  - evidence／evidence_tier: 全文判定說明與層級；不是原文的完整重製。
  - source_url／source_file／source_line: 原網址、保存的實際證據檔及適用時的歷史列號（1起算）。
  - reading_status／download_status／capture_note: 全文閱讀與下載分開；工具全文擷取與原HTML分開。
  - relative_name／relationship／degree／relative_office／inference_flag: G6關係、親等、公職、是否由共享親屬推論；未知姓名留空。
  - queries／capture／pass_type／status／note: 指定查詢、捕捉檔、輪次、決定及排除同名或非公職的理由。
  - lead_value／reason: 未採線索及未採理由，不能直接當作完成值。
  - path／sha256／bytes／role: 固定輸入檔身份、雜湊、位元組數與資料角色。
- Data path: `data/raw/taipei_kaohsiung_completion_20261108/input_manifest.json` 及其中每一精確路徑。
- Role: `raw`（網頁捕捉）／`documented_manual_review`（人工判定）；兩者有區分。
- Verification status: 本Notebook實際執行逐檔SHA與JSON可讀驗證；這不等於每個來源都可靠或完成全資料。
- SHA-256: manifest在下一格以文字常數固定；其餘逐檔SHA列於manifest及實際輸出表。
- Next processing notebook: `data/code/complete_taipei_kaohsiung_20261108.ipynb`

### 2026-11-10來源安全修正

林延鳳兩份HTML公開副本移除第三方網頁配置中的Mapbox token；原bytes私存於Git外。其他bytes、已存文章文字與候選人判讀不變。下載manifest的sha256現在指公開副本，original_capture_sha256保留最初捕捉指紋。capture_hygiene記錄修正前後SHA與範圍；input_manifest及本檔固定指紋同步更新。


In [1]:
from pathlib import Path
import json, hashlib, csv
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'data').is_dir()), None)
if ROOT is None: raise RuntimeError('請從本專案根目錄或其子目錄執行。')
RAW = ROOT/'data/raw/taipei_kaohsiung_completion_20261108'
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
manifest = RAW/'input_manifest.json'
assert sha(manifest) == 'eea59bf660497751639f93514f09b6b8c5a5e9c1f2d4a32150f1604f2432f7c6', 'input manifest版本改變，停止而不自動接受新SHA'
PINS = json.loads(manifest.read_text())
for item in PINS:
    f=ROOT/item['path']
    assert f.is_file() and sha(f)==item['sha256'], item['path']
print('輸入指紋通過：',len(PINS),'個固定檔案；manifest另行固定SHA。')

輸入指紋通過： 666 個固定檔案；manifest另行固定SHA。


## Change for Your Own Data
精確manifest與SHA固定，變動即停止，不從其他檔案替代。這份文件只讀取來源，不改寫來源。

In [2]:
inventory=pd.DataFrame(PINS)
captures=inventory[inventory.path.str.startswith('data/raw/taipei_kaohsiung_completion_20261108/')].copy()
json_files=captures[captures.path.str.endswith('.json')]
for f in json_files.path:json.loads((ROOT/f).read_text())
facts=pd.DataFrame(json.loads((RAW/'reviewed_facts.json').read_text())).fillna('')
print('新捕捉／閱讀紀錄檔案',len(captures),'；JSON可讀',len(json_files),'；人工觀察',len(facts))

新捕捉／閱讀紀錄檔案 519 ；JSON可讀 193 ；人工觀察 214


## Summary

### Sample

In [3]:
display(facts.head(10)[['candidate_name','variable','value','evidence_tier','source_file']])
display(captures.head(10))

,candidate_name,variable,value,evidence_tier,source_file
0,沈伯洋,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
1,李文,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
2,紀建漢,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
3,林揚庭,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
4,許甫,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
5,陳怡君,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
6,李惠暄,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
7,賴俊翰,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
8,林子揚,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...
9,應佳妤,G6_political_family,1,media_full_text,data/raw/taipei_kaohsiung_completion_20261108/...


,path,sha256,role,bytes
0,data/raw/taipei_kaohsiung_completion_20261108/...,5c3605e726418f1a884c8079c64baf9f088a8566fefc8f...,raw_capture_or_review_record,1114
1,data/raw/taipei_kaohsiung_completion_20261108/...,581e970db988379845f392ed0f25736d78cd286256bf25...,raw_capture_or_review_record,5270
2,data/raw/taipei_kaohsiung_completion_20261108/...,985e8cc16bd9d9f0e46d4d4df2090e984c0f0daf36a771...,raw_capture_or_review_record,5458
3,data/raw/taipei_kaohsiung_completion_20261108/...,2d2e6b58046a3f8967c2fe71a2f7789e37f06e898f19bb...,raw_capture_or_review_record,15681
4,data/raw/taipei_kaohsiung_completion_20261108/...,638337ed064280cf5c4166c0c3ba06c8a8f08b43a5a185...,raw_capture_or_review_record,2740
5,data/raw/taipei_kaohsiung_completion_20261108/...,965fbcd2fdcca68bea5c86e0686565a35aea973c80a721...,raw_capture_or_review_record,20001
6,data/raw/taipei_kaohsiung_completion_20261108/...,0ed3d7dd20513cea1c677cf49fd2c04eb0b25b1118bb7a...,raw_capture_or_review_record,2405
7,data/raw/taipei_kaohsiung_completion_20261108/...,f7bc3647ffb4fc566d3b64f70a8909c37bab97e47a5361...,raw_capture_or_review_record,198895
8,data/raw/taipei_kaohsiung_completion_20261108/...,08e88ae7f42303c7a311155913fee57a3fb2792a72d914...,raw_capture_or_review_record,3055
9,data/raw/taipei_kaohsiung_completion_20261108/...,f7bc3647ffb4fc566d3b64f70a8909c37bab97e47a5361...,raw_capture_or_review_record,198895


### Summary Stats

In [4]:
display(facts.groupby(['variable','value']).size().rename('observations').reset_index())
display(captures.assign(extension=captures.path.map(lambda x:Path(x).suffix)).groupby('extension').agg(files=('path','size'),bytes=('bytes','sum')))
display(pd.DataFrame(json.loads((RAW/'family_search_decisions.json').read_text())).groupby('status').size().rename('candidates'))
display(facts.dtypes.astype(str).rename('dtype').to_frame())
display(facts.eq('').sum().rename('empty_fields').to_frame())
assert not facts.duplicated(['candidate_key','variable']).any()
print('逐檔指紋、JSON格式、觀察鍵唯一性通過；來源品質限制見下。')

,variable,value,observations
0,G1_age_2026_minus_birth_year,24,1
1,G1_age_2026_minus_birth_year,31,2
2,G1_age_2026_minus_birth_year,35,1
3,G1_age_2026_minus_birth_year,39,3
4,G1_age_2026_minus_birth_year,42,1
5,G1_age_2026_minus_birth_year,49,1
6,G1_age_2026_minus_birth_year,52,1
7,G1_age_2026_minus_birth_year,54,1
8,G1_age_2026_minus_birth_year,58,1
9,G1_age_2026_minus_birth_year,70,1


,files,bytes
extension,,
.html,154,17526883
.json,193,4016001
.pdf,4,15086237
.txt,168,1388716


status
pending_scope_or_relation            5
positive_full_text                  37
searched_no_qualifying_evidence    139
Name: candidates, dtype: int64

,dtype
candidate_key,str
candidate_name,str
county_city,str
variable,str
value,str
evidence,str
evidence_tier,str
checked_date,str
source_url,str
source_file,str


,empty_fields
candidate_key,0
candidate_name,0
county_city,0
variable,0
value,5
evidence,0
evidence_tier,0
checked_date,0
source_url,139
source_file,0


逐檔指紋、JSON格式、觀察鍵唯一性通過；來源品質限制見下。


### 來源限制與後續

`capture_review_notes.json`列排除與失敗細節；`capture_hygiene.json`記公開捕捉憑證樣式清理，原bytes只在repo外私存。
下載manifest中的下載當時SHA和input_manifest中的保留版SHA不混用。存下錯頁是為了稽核，不能當成候選人證據。
正見與政精選的完整生日若無獨立原文支持，維持線索。原始登記PDF不載生日，無法佐證第三方生日。
G6搜尋負值為學生規則；原始reviewed_facts與family_search_decisions保留前一輪5例未決狀態，不能當最新值；新增g6_scope_decision與g6_followup_updates在處理Notebook覆疊後已無G6空白。G3政策fallback不等於已知最低學歷。
下一步讀本輪處理Notebook產出的缺口工作單；全資料未完成前不開始分群。